# Ingest Order Items Dataset file

1. read the file using spark dataframe reader API
- Define the Schema 
2. Add metadata columns
- source file 
- ingestion timestamp
3. write to the bronze delta table

In [0]:
%run ../01-common/01.bronze_helper

In [0]:
#Imports
from pyspark.sql.functions import col
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType, DoubleType

In [0]:
source_file="/Volumes/olist_catalog/landing/files/olist_order_items_dataset.csv"
table_name = "olist_catalog.bronze.order_items"

In [0]:
order_items_schema = StructType([
    StructField("order_id",StringType()),
    StructField("order_item_id",IntegerType()),
    StructField("product_id",StringType()),
    StructField("seller_id",StringType()),
    StructField("shipping_limit_date",TimestampType()),
    StructField("price",DoubleType()),
    StructField("freight_value",DoubleType())
])

In [0]:
order_items_df = (
    spark.read
    .format("csv")
    .option("header","True")
    .schema(order_items_schema)
    .load(source_file)
    )

In [0]:
order_items_df_final=add_ingestion_data(order_items_df)

In [0]:
display(order_items_df_final)

In [0]:
(
    order_items_df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
select * from olist_catalog.bronze.order_items